## 4. Textual Analysis: Embedding
Now, we will convert our article itself into numerical vector representation.

General we think about convert each word into vector using pre-trained model like fasttext, word2vec, glove etc. <br>
But we have 1000s of article with 1000s of words. This analysis is not possible. <br>

So, we will convert entire articles & summary into numerical vector representation using Transformer models. <br>
Specifically, for news article, we will use **BERT (Bidirectional Encoder Representations from Transformers)** model. <br>
For summary, we will use <b>GPT2 (Generative Pre-trained Transformer 2)</b> model. <br>

##### What NOT to Use for Textual Analysis:
1. ❌ TF-IDF + KMeans
-> lexical similarity ≠ semantic similarity <br>
2. ❌ LDA
-> topic modeling ≠ clustering <br>
3. ❌ Raw BERT CLS token
-> not trained for similarity <br>

Upon research i could find 3 different ways to convert text into numerical vector representation:
1. Raw BETR CLS token + GPT2 CLS token
2. AutoTokenizer + AutoModel pretrained conversion from Huggingface
3. Sentence-Transformer (SBERT)

In [1]:
import numpy as np
import pandas as pd

import torch

Length = pd.read_parquet("../Dataset/Clean/Combined_dataset.parquet")

Embedding = Length[['Content','Summary','char_count','sentence_count','word_count', 'unique_word_count']].copy()
Embedding

,Content,Summary,char_count,sentence_count,word_count,unique_word_count
0,New York police are concerned drones could bec...,Police have investigated criminals who have ri...,2906,17,483,241
1,By . Ryan Lipman . Perhaps Australian porn sta...,Porn star Angela White secretly filmed sex act...,1613,15,257,146
2,"This was, Sergio Garcia conceded, much like be...",American draws inspiration from fellow country...,5502,54,969,407
3,An Ebola outbreak that began in Guinea four mo...,World Health Organisation: 635 infections and ...,3666,33,550,271
4,By . Associated Press and Daily Mail Reporter ...,A sinkhole opened up at 5:15am this morning in...,3255,41,509,241
...,...,...,...,...,...,...
9796,Os na fydd modd dod o hyd i berchennog bydd y ...,Mae cwmni Tindle Newspapers wedi cadarnhau ei ...,1773,13,304,163
9797,Aston Villa have made a late charge to steal M...,Newcastle have made Dele Alli a primary transf...,1074,9,185,104
9798,"By . Nick Mcdermott, Science Reporter . PUBLIS...",National Childbirth Trust accused of championi...,3942,33,642,296
9799,(CNN) -- In anticipation of more flooding next...,Fargo spokeswoman says city has goal of fillin...,1256,10,207,121


##### 1. Raw BERT CLS token + GPT2 CLS token

In [ ]:
from transformers import BertTokenizer, BertModel, GPT2Tokenizer, GPT2Model

# Function to chunk text into smaller pieces
def chunk_text(text, tokenizer, max_length=512):
    tokens = tokenizer.tokenize(text)
    chunks = []
    for i in range(0, len(tokens), max_length - 2):  # Reserve space for [CLS] and [SEP]
        chunk = tokens[i:i + max_length - 2]
        chunk = ['[CLS]'] + chunk + ['[SEP]']
        chunks.append(tokenizer.convert_tokens_to_ids(chunk))
    return chunks

# Function to get embeddings from model
def get_embeddings(chunks, model, device='cpu'):
    model.to(device)
    model.eval() # evaluation mode
    
    embeddings = []
    with torch.no_grad():
        for chunk in chunks:
            input_ids = torch.tensor([chunk]).to(device)
            outputs = model(input_ids)
            # Use mean pooling of last hidden state
            embedding = outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
            embeddings.append(embedding)
    # Average embeddings across chunks
    if embeddings:
        return np.mean(embeddings, axis=0)
    else:
        return np.zeros(model.config.hidden_size)

In [ ]:
# Load models and tokenizers
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased')
gpt2_tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token  # Set pad token
gpt2_model = GPT2Model.from_pretrained('gpt2')

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

Embedding = Embedding.head(10)  # For testing, process only first 10 rows

# Assuming Embedding is the dataframe with 'Content' and 'Summary' columns
# Add new columns for embeddings
Embedding['content_embedding'] = None
Embedding['summary_embedding'] = None

for idx, row in Embedding.iterrows():
    # Chunk and embed content with BERT
    content_chunks = chunk_text(row['Content'], bert_tokenizer)
    content_emb = get_embeddings(content_chunks, bert_model, device)
    Embedding.at[idx, 'content_embedding'] = content_emb
    
    # Chunk and embed summary with GPT2
    summary_chunks = chunk_text(row['Summary'], gpt2_tokenizer)
    summary_emb = get_embeddings(summary_chunks, gpt2_model, device)
    Embedding.at[idx, 'summary_embedding'] = summary_emb

This is one of the most common ways to convert text/article into numerical vector representation but there are some problems with this aaproach.
1. Raw bert and GPT-2 are trained for next word prediction & masked language modeling while our tasks are clustering, semantic similarity etc.
2. These large model are very slow as they work token by token which are then averaged out for final result -> not accurate
3. This approach is resource intensive - GPU which we don't have.

##### 2. Huggingface AutoTokenizer + AutoModel pretrained conversion
This is inheriently similar approach just using model loader. This still faces the same issues as above.


##### 3. Sentence-Transformer (SBERT)
This is a pretrained model tuned for semantic search applications. <br>
This is the most suitable approach for our task.

In [ ]:
# i) Loading model
from sentence_transformers import SentenceTransformer
import torch
import nltk
import spacy
from typing import List, Optional

In [ ]:
## ii) Configuring model
model_name = 'all-MiniLM-L6-v2'
Max_sent_per_chunk = 5
Embedding_batch_size = 32

In [ ]:
## iii) Convert text into chunk
class TextChucker:
    def __init__(self, max_sent:int):
        self.max_sentence = max_sent
        self.nlp = spacy.load("en_core_web_sm", disable=["ner","tagger"])
        self.nlp.add_pipe("sentencizer") # required for doc.sents
        
    def chunk(self, text:str, max_sent:Optional[int]=None, num_chunks:Optional[int]=None) -> List[str]:
        doc = self.nlp(text)
        sent = [s.text.strip() for s in doc.sents if s.text.strip()]
        
        if not sent:
            print("No sentences found in the text.")
            return []
        total_sent = len(sent)
        
        # Mode-1 number of chunks
        if num_chunks is not None:
            if num_chunks <= 0:
                raise ValueError("num_chunks must be a positive integer.")
            
            num_chunks = min(num_chunks, total_sent)
            sent_per_chunk = np.ceil(total_sent / num_chunks).astype(int)
        # Mode-2 number of sentences per chunk
        else:
            sent_per_chunk = max_sent if max_sent is not None else self.max_sentence
            if sent_per_chunk <= 0 or sent_per_chunk is None:
                raise ValueError("max_sent must be a valid positive integer.")
        
        # Building chunks
        chucks = []
        for i in range(0, len(sent), sent_per_chunk):
            chunk = " ".join(sent[i:i+sent_per_chunk])
            chucks.append(chunk)
            
        return chucks
    
    def chunk_batch(self, texts:List[str], max_sent:Optional[int]=None, num_chunks:Optional[int]=None) -> List[List[str]]:
        """
        Chunk a batch of articles.

        Returns:
            List[List[str]] -> chunks per article
        """
        if isinstance(texts, str):
            texts = [texts]
        # Basic checks
        if max_sent is not None and (max_sent <= 0):
            raise ValueError("max_sent must be a positive integer.")
        if num_chunks is not None and (num_chunks <= 0):
            raise ValueError("num_chunks must be a positive integer.")
        
        docs = self.nlp.pipe(texts, batch_size=64)
        
        # Building chunks
        all_chucks = []
        for doc in docs:
            sent = [s.text.strip() for s in doc.sents if s.text.strip()]
            total_sent = len(sent)
            
            if total_sent == 0:
                all_chucks.append([])
                continue
            
            # Mode-1 number of chunks
            if num_chunks is not None:
                num_chunks = min(num_chunks, total_sent)
                sent_per_chunk = int(np.ceil(total_sent / num_chunks))
            # Mode-2 number of sentences per chunk
            else:
                sent_per_chunk = max_sent if max_sent is not None else self.max_sentence
                
            chunks = [" ".join(sent[i:i+sent_per_chunk])
                for i in range(0, len(sent), sent_per_chunk)]
            
            all_chucks.append(chunks)
            
        return all_chucks
    
# Testing
Chucker_test = TextChucker(3)
res = Chucker_test.chunk_batch(Embedding.loc[12:16,'Content'], num_chunks=6)
for j in res:
    print(*j, end="\n ----------------------------------------- \n")

In [ ]:
## iv) Convert text into embedding
class SentenceEmbedding:
    _shared_model = None
    
    def __init__(self, model_name:str):
        if SentenceEmbedding._shared_model is None:
            SentenceEmbedding._shared_model = SentenceTransformer(model_name)
        self.model = SentenceEmbedding._shared_model
        
    def Embed_chunks(self, chunks:List[str] | str, batch_size:int = 64, weighted:bool=True) -> np.ndarray:
        """
        Convert list of text chunks into a single embedding vector.
         - If `weighted` is True, it weights the chunk embeddings by their word count.
         - If `weighted` is False, it simply averages the chunk embeddings.
        """
        if isinstance(chunks, str):
            chunks = [chunks]
        encode = self.model.encode(chunks, batch_size=batch_size, convert_to_numpy=True, normalize_embeddings=False, show_progress_bar=False)
        
        # Now, Combining all Chunks into single sentence for each Article
        if weighted:
            weight = np.array([len(c.split()) for c in chunks], dtype='float32')
            weight /= weight.sum()
        
            pooled = np.average(encode, axis=0, weights=weight)
        else:
            pooled = np.average(encode, axis=0)
        # Normalize with L2 norm
        norm = np.linalg.norm(pooled)
        
        return pooled/norm if norm>0 else pooled

sent_embedding_test = SentenceEmbedding(model_name=model_name)

In [ ]:
Encoded = sent_embedding_test.Embed_chunks(res)
print(Encoded.shape)

In [ ]:
## v) Embedding pipeline for entire dataset
class ArticleEmbeddingPipeline:
    def __init__(self, chunker, embedder):
        self.chunker = chunker
        self.embedder = embedder
        
    def embedded_articles(self, articles:List[str]) -> List[np.ndarray]:
        chunked_article = self.chunker.chunk_batch(articles, max_sent = Max_sent_per_chunk)
        
        article_embeddings = []
        
        for chunk in chunked_article:
            if not chunk:
                print("Empty chunk found, skipping embedding.")
                dim = self.embedder.model.get_sentence_embedding_dimension()
                article_embeddings.append(np.zeros(dim))
            else:
                test_chunks = self.embedder.Embed_chunks(chunk, weighted=True)
                article_embeddings.append(test_chunks.astype("float32"))
        
        return np.array(article_embeddings)
    
art_embedder = ArticleEmbeddingPipeline(Chucker_test, sent_embedding_test)

In [ ]:
import tqdm

text_cols = Embedding['Content']
batch_sz = 32

## Creating Embedding for all articles
Chunker = TextChucker(Max_sent_per_chunk)
Embedder = SentenceEmbedding(model_name)
pipeline = ArticleEmbeddingPipeline(Chunker, Embedder)

# Generate embedding [Batched]
all_embedding = []
for i in tqdm.tqdm(range(0, len(text_cols), batch_sz), desc="Embedding articles"):
    batch_texts = text_cols[i:i+batch_sz]
    
    batch_emb = pipeline.embedded_articles(batch_texts)
    all_embedding.extend(batch_emb)
    
print("Length of embedding: ",len(all_embedding))
print("Sample: ", all_embedding[0])

In [ ]:
Embedding['Embedding'] = all_embedding

# Saving these embeddings
# Embedding.to_parquet("../Dataset/Clean/Embeddings.parquet", index=False)

This is methodically correct. But this embedding is too slow. <br>
Now, we will optimize it for speed.

In [ ]:
import numpy as np
import pandas as pd
import spacy
from sentence_transformers import SentenceTransformer
from typing import List
from tqdm import tqdm


class ArticleEmbeddingEngine:
    """
    End-to-end, high-performance article embedding engine.
    Converts each article into a single normalized vector embedding.
    """

    def __init__(
        self,
        model_name: str = "all-MiniLM-L6-v2",
        max_sentences_per_chunk: int = 5,
        spacy_batch_size: int = 64,
        embedding_batch_size: int = 64,
    ):
        self.max_sentences = max_sentences_per_chunk
        self.spacy_batch_size = spacy_batch_size
        self.embedding_batch_size = embedding_batch_size

        # Load SBERT once
        self.embedder = SentenceTransformer(model_name)

        # Load spaCy once (sentence segmentation only)
        self.nlp = spacy.load(
            "en_core_web_sm",
            disable=["ner", "tagger", "lemmatizer"]
        )
        self.nlp.enable_pipe("senter")

    # ---------------------------
    # Internal helpers
    # ---------------------------
    def _chunk_documents(self, texts: List[str]) -> List[List[str]]:
        """
        Sentence split + chunk documents in batch.
        """
        docs = self.nlp.pipe(texts, batch_size=self.spacy_batch_size)

        all_chunks = []
        for doc in docs:
            sentences = [s.text.strip() for s in doc.sents if s.text.strip()]
            chunks = [
                " ".join(sentences[i:i + self.max_sentences])
                for i in range(0, len(sentences), self.max_sentences)
            ]
            all_chunks.append(chunks)

        return all_chunks

    def _weighted_pool(self, embeddings: np.ndarray, chunks: List[str]) -> np.ndarray:
        """
        Length-weighted mean pooling + normalization.
        """
        weights = np.array([len(c.split()) for c in chunks], dtype=np.float32)
        weights /= weights.sum()

        pooled = np.average(embeddings, axis=0, weights=weights)

        norm = np.linalg.norm(pooled)
        return pooled / norm if norm > 0 else pooled

    # ---------------------------
    # Public API
    # ---------------------------

    def embed_dataframe(
        self,
        df: pd.DataFrame,
        text_column: str = "Content",
    ) -> pd.DataFrame:
        """
        Converts all articles in a DataFrame to embeddings
        and adds them as a new column.
        """

        texts = df[text_column].fillna("").tolist()

        # Step 1: Chunk all documents (batched spaCy)
        chunked_docs = self._chunk_documents(texts)

        embeddings_out = []

        # Step 2: Embed + pool per article
        for chunks in tqdm(chunked_docs, desc="Embedding articles"):
            if not chunks:
                embeddings_out.append(np.zeros(384, dtype=np.float32))
                continue

            chunk_embeddings = self.embedder.encode(
                chunks,
                batch_size=self.embedding_batch_size,
                convert_to_numpy=True,
                show_progress_bar=False,
                normalize_embeddings=False,
            )

            article_embedding = self._weighted_pool(chunk_embeddings, chunks)
            embeddings_out.append(article_embedding.astype(np.float32))

        return embeddings_out

In [ ]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer


class FastArticleEmbeddingEngine:
    """
    High-performance document embedding engine using Sentence-Transformers.
    """

    def __init__(
        self,
        model_name: str = "all-MiniLM-L6-v2",
        batch_size: int = 64,
        max_seq_length: int = 512,
    ):
        self.batch_size = batch_size

        self.model = SentenceTransformer(model_name)
        self.model.max_seq_length = max_seq_length  # ✅ correct place

    def embed_dataframe(
        self,
        df: pd.DataFrame,
        text_column: str = "Content",
    ) -> List[np.ndarray]:

        texts = df[text_column].fillna("").astype(str).tolist()

        embeddings = self.model.encode(
            texts,
            batch_size=self.batch_size,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,  # cosine-ready
        )

        return embeddings.astype(np.float32).tolist()


In [ ]:
engine = FastArticleEmbeddingEngine()
Length['Embedding'] = engine.embed_dataframe(Length, text_column='Content')

Now, we have an even better and updated model for embedding articles: <br>
- "all-mpnet-base-v2" -> Fast, good, 5x slower
- "intfloat/e5-large-v2" -> best quality, slow
- "BAAI/bge-small-en" -> best quality

In [ ]:
# Saving these embeddings
# Length.to_parquet("../Dataset/Clean/Dataset_with_embeddings.parquet", index=False)

#### NOTE:
Before any task, we will normalize these embeddings. 

### Q. Why normalize embeddings?
Normalization is done in vector-based tasks because it transforms your embeddings into Unit Vectors (length of 1), aka standard length.

Here are the three main reasons why we do this in your project: <br>
##### 1. Dot Product = Cosine Similarity <br>
- Cosine Similarity measures the angle between two vectors, regardless of their size. It's the gold standard for "meaning" similarity.
- The formula for Cosine Similarity is: 

$$ (A · B) / (||A|| * ||B||) $$
- If we normalize vectors first (||A|| = 1 and ||B|| = 1), the denominator becomes 1.

Result: You can calculate the similarity using a simple Dot Product, which is significantly faster. Your FAISS index uses **IndexFlatIP** (Inner Product) specifically to take advantage of this speed.

##### 2. Focus on "Direction" over "Magnitude"
- In NLP, we usually care about the topic or intent of a sentence (its direction in space), not how "loud" or "long" it is (its magnitude).
- Without normalization, a very long article might have a higher magnitude simply because it contains more words, even if it has the same meaning as a short one. <br> Normalizing ensures that every article, regardless of word count, has an equal "vote" in the vector space.

##### 3. Mathematical Stability for KMeans and BERTopic
- Clustering algorithms like KMeans typically use Euclidean Distance which for normalized vector becomes mathematically equivalent (they will rank results in the same order).
- This forces the clusters to form based on the "meaningful" angle of the data rather than being skewed by outliers with large vector values.

> In summary: We normalize to make the math faster (Inner Product), the comparisons fairer (length doesn't matter), and the clusters tighter.